# 🔪 법령 문서 청킹 전략 만들기

## 🎯 실습 목표
- 법령 문서의 구조를 이해하고 효과적으로 분할하는 **함수**들을 직접 만들어보기
- 각 함수를 테스트하면서 동작 확인하기
- 마지막에 완성된 **클래스**를 실행해보기

---

## 목적
통합된 법령 문서를 AI가 효율적으로 검색하고 이해할 수 있는 작은 단위(청크)로 분할하기 위함입니다. 법령 문서는 조(條), 항(項), 호(號) 등의 고유한 구조를 가지고 있어, 이를 고려한 지능적인 분할이 필요합니다. 단순히 글자 수나 토큰 수로만 자르면 문맥이 끊겨 AI가 정확한 답변을 할 수 없기 때문에, 법령의 논리적 구조를 유지하면서 적절한 크기로 나누는 것이 핵심입니다.

## 기능
통합 JSON 파일의 각 텍스트 블록을 토큰 수(약 800토큰)를 기준으로 분할하되, 법령의 조(條) 단위를 최대한 보존합니다. 한 조가 너무 길면 항(項) 단위로 세분화하고, 그래도 길면 토큰 단위로 강제 분할합니다. 청크 간 오버랩(200토큰)을 적용하여 문맥의 연속성을 보장하며, tiktoken 라이브러리를 사용해 정확한 토큰 수를 계산합니다.

## 결과물
각 청크는 고유 ID, 텍스트 내용, 메타데이터(출처 문서, 페이지, 토큰 수, 오버랩 여부)를 포함하는 JSON 파일이 생성됩니다. 이 청크들은 다음 단계인 임베딩 작업의 입력이 되며, 각 청크가 벡터로 변환되어 벡터 데이터베이스에 저장됩니다. 사용자가 질문하면 AI는 이 청크들 중에서 관련된 내용을 빠르게 검색하여 정확한 답변을 생성할 수 있습니다.

---


## 📦 1. 라이브러리 임포트

In [1]:
# 필요한 라이브러리 설치 (처음 한번만 실행)
import sys
!{sys.executable} -m pip install tiktoken --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [tiktoken]1/5 [regex]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/jupyter/kernel-py312-env/bin/python -m pip install --upgrade pip


In [2]:
import re
import json
import os
from typing import List, Dict
import tiktoken

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


## 📦 2. 테스트용 데이터 로드

In [3]:
# 테스트용 통합 JSON 파일 경로
unified_json_path = "data/processed/construction_law_unified.json"

if os.path.exists(unified_json_path):
    print(f"✅ 파일 확인: {unified_json_path}")
    
    with open(unified_json_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    print(f"\n메타데이터:")
    print(f"  - 문서 수: {test_data['metadata']['total_documents']}개")
    print(f"  - 텍스트 블록: {len(test_data['text_blocks'])}개")
    
    # 샘플 블록 확인
    if test_data['text_blocks']:
        sample = test_data['text_blocks'][0]
        print(f"\n샘플 블록:")
        print(f"  - Block ID: {sample['block_id']}")
        print(f"  - Doc Name: {sample['doc_name']}")
        print(f"  - Text (앞 100자): {sample['text'][:100]}...")
else:
    print(f"❌ 파일 없음: {unified_json_path}")

✅ 파일 확인: data/processed/construction_law_unified.json

메타데이터:
  - 문서 수: 4개
  - 텍스트 블록: 1000개

샘플 블록:
  - Block ID: block_00001
  - Doc Name: (AURI)해석례로 읽는 건축법.pdf
  - Text (앞 100자): 해석례로
읽는
건축법2024
Q&A
Architecture & Urban Research Institute...


---

## 🔧 Step 1: 토큰 인코더 초기화 및 토큰 카운트

### ToDo #1: `init_encoder()` 및 `count_tokens()` 함수 만들기

**목표**: tiktoken 인코더를 초기화하고 텍스트의 토큰 수를 세는 함수 만들기

**힌트**:
- `tiktoken.encoding_for_model(model_name)`: 모델에 맞는 인코더 생성
- `encoder.encode(text)`: 텍스트를 토큰으로 변환
- `len(tokens)`: 토큰 개수

In [4]:
def init_encoder(model: str = "gpt-4"):
    """
    tiktoken 인코더를 초기화합니다.
    
    Args:
        model: 모델명 (기본값: gpt-4)
    
    Returns:
        tiktoken.Encoding: 인코더 객체
    """
    # ToDo #1-1: 여기에 코드를 작성하세요
    # tiktoken.encoding_for_model(model)로 인코더 생성
    pass

def count_tokens(text: str, encoder) -> int:
    """
    텍스트의 토큰 수를 계산합니다.
    
    Args:
        text: 입력 텍스트
        encoder: tiktoken 인코더
    
    Returns:
        int: 토큰 개수
    """
    # ToDo #1-2: 여기에 코드를 작성하세요
    # 1. encoder.encode(text)로 토큰 리스트 생성
    # 2. len()으로 토큰 개수 반환
    pass

### ✅ 테스트 #1: 인코더 초기화 및 토큰 카운트

In [5]:
# 인코더 초기화
encoder = init_encoder("gpt-4")
print("✅ 인코더 초기화 완료")

# 토큰 카운트 테스트
test_text = "제1조(목적) 이 법은 건축물의 대지·구조·설비 기준 및 용도 등을 정하여 건축물의 안전·기능·환경 및 미관을 향상시킴으로써 공공복리의 증진에 이바지함을 목적으로 한다."
token_count = count_tokens(test_text, encoder)

print(f"\n테스트 텍스트: {test_text}")
print(f"토큰 수: {token_count}")

✅ 인코더 초기화 완료

테스트 텍스트: 제1조(목적) 이 법은 건축물의 대지·구조·설비 기준 및 용도 등을 정하여 건축물의 안전·기능·환경 및 미관을 향상시킴으로써 공공복리의 증진에 이바지함을 목적으로 한다.
토큰 수: 100


---

## 🔧 Step 2: 토큰 단위 강제 분할

### ToDo #2: `split_by_tokens()` 함수 만들기

**목표**: 텍스트를 지정된 토큰 수로 강제 분할하기 (최후의 수단)

**힌트**:
- `encoder.encode(text)`: 텍스트를 토큰 리스트로 변환
- `tokens[i:i+max_tokens]`: 슬라이싱으로 토큰 분할
- `encoder.decode(tokens)`: 토큰을 다시 텍스트로 변환
- `range(0, len(tokens), max_tokens)`: max_tokens씩 건너뛰며 순회

In [6]:
def split_by_tokens(text: str, encoder, max_tokens: int) -> List[str]:
    """
    토큰 단위로 텍스트를 강제 분할합니다.
    
    Args:
        text: 입력 텍스트
        encoder: tiktoken 인코더
        max_tokens: 청크당 최대 토큰 수
    
    Returns:
        List[str]: 분할된 텍스트 리스트
    """
    # ToDo #2: 여기에 코드를 작성하세요
    # 1. encoder.encode(text)로 전체 토큰 리스트 생성
    # 2. range(0, len(tokens), max_tokens)로 순회
    # 3. 각 구간의 토큰 슬라이싱: tokens[i:i+max_tokens]
    # 4. encoder.decode()로 텍스트로 변환
    # 5. 리스트에 추가하여 반환
    pass

### ✅ 테스트 #2: 토큰 단위 분할

In [7]:
# 긴 텍스트 생성 (테스트용)
long_text = test_text * 50  # 반복해서 긴 텍스트 만들기
print(f"원본 텍스트 토큰 수: {count_tokens(long_text, encoder)}")

# 100 토큰씩 분할
chunks = split_by_tokens(long_text, encoder, 100)

print(f"\n✅ 분할 결과:")
print(f"청크 수: {len(chunks)}개")
print(f"\n첫 번째 청크 (토큰 수: {count_tokens(chunks[0], encoder)}):")
print(chunks[0][:200] + "...")

원본 텍스트 토큰 수: 5000

✅ 분할 결과:
청크 수: 50개

첫 번째 청크 (토큰 수: 100):
제1조(목적) 이 법은 건축물의 대지·구조·설비 기준 및 용도 등을 정하여 건축물의 안전·기능·환경 및 미관을 향상시킴으로써 공공복리의 증진에 이바지함을 목적으로 한다....


---

## 🔧 Step 3: 조(條) 단위 분할 (완성 코드)

### 설명

법령의 **"제○조"** 패턴을 인식하여 조 단위로 분할하는 함수입니다.

**작동 원리**:
1. 정규식으로 "제1조", "제2조", "제5조의2" 등의 패턴 인식
2. 조 제목과 내용을 함께 묶어서 처리
3. 토큰 수를 확인하면서 청크 조합
4. 조가 너무 크면 토큰 단위로 강제 분할

In [8]:
def split_by_article(text: str, encoder, max_tokens: int) -> List[str]:
    """
    조(條) 단위로 텍스트를 분할합니다.
    
    Args:
        text: 입력 텍스트
        encoder: tiktoken 인코더
        max_tokens: 청크당 최대 토큰 수
    
    Returns:
        List[str]: 분할된 텍스트 리스트
    """
    # 제○조 패턴으로 분할
    parts = re.split(r'(제\d+조(?:의\d+)?)', text)
    
    chunks = []
    current_chunk = ""
    current_tokens = 0
    
    i = 0
    while i < len(parts):
        part = parts[i].strip()
        
        if not part:
            i += 1
            continue
        
        # 제○조 패턴인지 확인
        if re.match(r'제\d+조(?:의\d+)?$', part):
            article = part
            content = ""
            
            # 다음 파트(내용) 가져오기
            if i + 1 < len(parts):
                content = parts[i + 1].strip()
            
            full_text = article + " " + content
            text_tokens = count_tokens(full_text, encoder)
            
            # 조 전체가 max_tokens 이하
            if text_tokens <= max_tokens:
                # 현재 청크에 추가 가능한지 확인
                if current_tokens + text_tokens <= max_tokens:
                    current_chunk += (" " if current_chunk else "") + full_text
                    current_tokens += text_tokens
                else:
                    # 현재 청크 저장하고 새로 시작
                    if current_chunk:
                        chunks.append(current_chunk)
                    current_chunk = full_text
                    current_tokens = text_tokens
            else:
                # 조가 너무 크면 토큰 단위로 강제 분할
                if current_chunk:
                    chunks.append(current_chunk)
                
                token_chunks = split_by_tokens(full_text, encoder, max_tokens)
                chunks.extend(token_chunks)
                
                current_chunk = ""
                current_tokens = 0
            
            i += 2  # 조 제목과 내용 건너뛰기
        else:
            # 일반 텍스트 처리
            part_tokens = count_tokens(part, encoder)
            
            if current_tokens + part_tokens <= max_tokens:
                current_chunk += (" " if current_chunk else "") + part
                current_tokens += part_tokens
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = part
                current_tokens = part_tokens
            
            i += 1
    
    # 마지막 청크 저장
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks

print("✅ split_by_article() 함수 정의 완료")

✅ split_by_article() 함수 정의 완료


### ✅ 테스트 #3: 조 단위 분할

In [9]:
# 샘플 법령 텍스트
sample_law = """
제1조(목적) 이 법은 건축물의 대지·구조·설비 기준 및 용도 등을 정하여 건축물의 안전·기능·환경 및 미관을 향상시킴으로써 공공복리의 증진에 이바지함을 목적으로 한다.
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. "건축물"이란 토지에 정착하는 공작물 중 지붕과 기둥 또는 벽이 있는 것과 이에 딸린 시설물을 말한다.
2. "건축"이란 건축물을 신축·증축·개축·재축하거나 건축물을 이전하는 것을 말한다.
제3조(적용 제외) 이 법은 다음 각 호의 어느 하나에 해당하는 건축물에는 적용하지 아니한다.
"""

print(f"원본 토큰 수: {count_tokens(sample_law, encoder)}")

# 150 토큰 기준으로 분할
article_chunks = split_by_article(sample_law, encoder, 150)

print(f"\n✅ 조 단위 분할 결과:")
print(f"청크 수: {len(article_chunks)}개\n")

for i, chunk in enumerate(article_chunks, 1):
    tokens = count_tokens(chunk, encoder)
    print(f"[청크 {i}] (토큰: {tokens})")
    print(chunk[:150] + "...\n")

원본 토큰 수: 295

✅ 조 단위 분할 결과:
청크 수: 3개

[청크 1] (토큰: 100)
제1조 (목적) 이 법은 건축물의 대지·구조·설비 기준 및 용도 등을 정하여 건축물의 안전·기능·환경 및 미관을 향상시킴으로써 공공복리의 증진에 이바지함을 목적으로 한다....

[청크 2] (토큰: 148)
제2조 (정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. "건축물"이란 토지에 정착하는 공작물 중 지붕과 기둥 또는 벽이 있는 것과 이에 딸린 시설물을 말한다.
2. "건축"이란 건축물을 신축·증축·개축·재축하거나 건축물을 이전하는 것을 말한다....

[청크 3] (토큰: 46)
제3조 (적용 제외) 이 법은 다음 각 호의 어느 하나에 해당하는 건축물에는 적용하지 아니한다....



---

## 🔧 Step 4: 통합 JSON 처리 및 청킹 (완성 코드)

### 설명

통합 JSON 파일의 모든 텍스트 블록을 청킹하는 함수입니다.

**작동 원리**:
1. JSON 파일 로드 후 `text_blocks` 추출
2. 각 블록에 대해 `split_by_article()` 호출
3. 청크마다 고유 ID와 메타데이터 추가

In [10]:
def process_unified_json(json_path: str, encoder, chunk_size: int = 800) -> List[Dict]:
    """
    통합 JSON 파일을 처리하여 청크를 생성합니다.
    
    Args:
        json_path: 통합 JSON 파일 경로
        encoder: tiktoken 인코더
        chunk_size: 청크 최대 토큰 수
    
    Returns:
        List[Dict]: 청크 리스트
    """
    print(f"\n📖 JSON 로드: {json_path}")
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    text_blocks = data.get('text_blocks', [])
    print(f"✓ {len(text_blocks)}개 텍스트 블록")
    
    all_chunks = []
    chunk_counter = 1
    
    print(f"\n🔪 청킹 시작...")
    
    for idx, block in enumerate(text_blocks, 1):
        text = block['text']
        
        # 조 단위 청킹
        text_chunks = split_by_article(text, encoder, chunk_size)
        
        for chunk_text in text_chunks:
            # 청크 데이터 구성
            chunk = {
                "chunk_id": f"chunk_{chunk_counter:05d}",
                "content": chunk_text,
                "metadata": {
                    "doc_id": block["doc_id"],
                    "doc_name": block["doc_name"],
                    "page": block.get("page", 0),
                    "chunk_tokens": count_tokens(chunk_text, encoder)
                }
            }
            all_chunks.append(chunk)
            chunk_counter += 1
        
        if idx % 50 == 0:
            print(f"  진행: {idx}/{len(text_blocks)} 블록...")
    
    print(f"✅ 총 {len(all_chunks)}개 청크 생성\n")
    return all_chunks

print("✅ process_unified_json() 함수 정의 완료")

✅ process_unified_json() 함수 정의 완료


### ✅ 테스트 #4: 통합 JSON 청킹

In [11]:
# 청킹 실행
chunks = process_unified_json(unified_json_path, encoder, chunk_size=800)

print(f"청킹 결과 요약:")
print(f"  - 총 청크 수: {len(chunks)}개")

if chunks:
    print(f"\n첫 번째 청크 샘플:")
    sample_chunk = chunks[0]
    print(f"  - Chunk ID: {sample_chunk['chunk_id']}")
    print(f"  - Tokens: {sample_chunk['metadata']['chunk_tokens']}")
    print(f"  - Doc: {sample_chunk['metadata']['doc_name']}")
    print(f"  - Content (앞 200자): {sample_chunk['content'][:200]}...")


📖 JSON 로드: data/processed/construction_law_unified.json
✓ 1000개 텍스트 블록

🔪 청킹 시작...
  진행: 50/1000 블록...
  진행: 100/1000 블록...
  진행: 150/1000 블록...
  진행: 200/1000 블록...
  진행: 250/1000 블록...
  진행: 300/1000 블록...
  진행: 350/1000 블록...
  진행: 400/1000 블록...
  진행: 450/1000 블록...
  진행: 500/1000 블록...
  진행: 550/1000 블록...
  진행: 600/1000 블록...
  진행: 650/1000 블록...
  진행: 700/1000 블록...
  진행: 750/1000 블록...
  진행: 800/1000 블록...
  진행: 850/1000 블록...
  진행: 900/1000 블록...
  진행: 950/1000 블록...
  진행: 1000/1000 블록...
✅ 총 1853개 청크 생성

청킹 결과 요약:
  - 총 청크 수: 1853개

첫 번째 청크 샘플:
  - Chunk ID: chunk_00001
  - Tokens: 28
  - Doc: (AURI)해석례로 읽는 건축법.pdf
  - Content (앞 200자): 해석례로
읽는
건축법2024
Q&A
Architecture & Urban Research Institute...


---

## 🔧 Step 5: 청크 간 오버랩 적용 (완성 코드)

### 설명

청크 간 오버랩을 적용하여 문맥 연속성을 보장하는 함수입니다.

**작동 원리**:
1. 각 청크에 다음 청크의 앞부분(overlap 토큰)을 미리보기로 추가
2. 같은 문서(`doc_id`)의 청크끼리만 오버랩 적용
3. 오버랩 텍스트는 구분자로 표시

In [12]:
def apply_overlap(chunks: List[Dict], encoder, overlap: int = 200) -> List[Dict]:
    """
    청크 간 오버랩을 적용합니다.
    
    Args:
        chunks: 청크 리스트
        encoder: tiktoken 인코더
        overlap: 오버랩 토큰 수
    
    Returns:
        List[Dict]: 오버랩이 적용된 청크 리스트
    """
    if not chunks or overlap == 0:
        return chunks
    
    print(f"🔗 오버랩 적용 (overlap: {overlap} 토큰)...")
    
    overlapped_chunks = []
    
    for i, chunk in enumerate(chunks):
        content = chunk["content"]
        has_overlap = False
        
        # 다음 청크가 있고, 같은 문서인 경우
        if i < len(chunks) - 1:
            next_chunk = chunks[i + 1]
            
            if next_chunk["metadata"]["doc_id"] == chunk["metadata"]["doc_id"]:
                next_content = next_chunk["content"]
                
                # 다음 청크의 앞부분을 토큰으로 추출
                tokens = encoder.encode(next_content)
                
                if len(tokens) > overlap:
                    overlap_tokens = tokens[:overlap]
                    overlap_text = encoder.decode(overlap_tokens)
                    content += f"\n\n[다음 내용 미리보기]\n{overlap_text}..."
                    has_overlap = True
        
        # 오버랩이 적용된 청크 생성
        overlapped_chunks.append({
            **chunk,
            "content": content,
            "metadata": {
                **chunk["metadata"],
                "has_overlap": has_overlap,
                "chunk_tokens": count_tokens(content, encoder)
            }
        })
    
    print(f"✅ 오버랩 적용 완료\n")
    return overlapped_chunks

print("✅ apply_overlap() 함수 정의 완료")

✅ apply_overlap() 함수 정의 완료


### ✅ 테스트 #5: 오버랩 적용

In [13]:
# 오버랩 적용
final_chunks = apply_overlap(chunks, encoder, overlap=200)

print(f"오버랩 적용 결과:")
print(f"  - 총 청크 수: {len(final_chunks)}개")

# 오버랩이 있는 청크 찾기
with_overlap = [c for c in final_chunks if c['metadata'].get('has_overlap')]
print(f"  - 오버랩 있는 청크: {len(with_overlap)}개")

if with_overlap:
    print(f"\n오버랩 샘플 (마지막 200자):")
    print(with_overlap[0]['content'][-200:])

🔗 오버랩 적용 (overlap: 200 토큰)...
✅ 오버랩 적용 완료

오버랩 적용 결과:
  - 총 청크 수: 1853개
  - 오버랩 있는 청크: 1532개

오버랩 샘플 (마지막 200자):
3장 건축물의 유지와 관리 38 제38조 건축물대장 38 제39조 등기촉탁 66
제4장 건축물의 대지와 도로 69 제43조 공개 공지 등의 확보 69 제44조 대지와 도로의 관계 76 제45조 도로의 지정·폐지 또는 변경 82
제5장 건축물의 구조 및 재료 등 93 제48조 구조내력 등 93 제49조 건축물의 피난시설 및 용도제한 등 99 제50조 건...


---

## 🔧 Step 6: 청크 저장

### ToDo #3: `save_chunks()` 함수 만들기

**목표**: 청크를 JSON 파일로 저장하기

**힌트**:
- `os.path.dirname()`: 디렉토리 경로 추출
- `os.makedirs()`: 디렉토리 생성
- `json.dump()`: JSON 저장 (ensure_ascii=False, indent=2)

In [14]:
def save_chunks(chunks: List[Dict], output_path: str):
    """
    청크를 JSON 파일로 저장합니다.
    
    Args:
        chunks: 청크 리스트
        output_path: 출력 파일 경로
    """
    # ToDo #3: 여기에 코드를 작성하세요
    # 1. os.path.dirname()으로 디렉토리 추출
    # 2. os.makedirs()로 디렉토리 생성
    # 3. JSON 저장 (ensure_ascii=False, indent=2)
    pass

### ✅ 테스트 #6: 청크 저장

In [15]:
# 청크 저장
output_path = "data/chunks/construction_law_chunks.json"
save_chunks(final_chunks, output_path)

# 저장 확인
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"✅ 저장 확인: {file_size:,} bytes")

💾 청크 저장: data/chunks/construction_law_chunks.json
  - 청크 수: 1853개
✅ 저장 확인: 3,477,123 bytes


---

## 🎓 Step 7: 완성된 클래스 확인 및 실행

### 설명

위에서 만든 모든 함수들이 하나의 클래스로 통합된 형태입니다.
이제 이 클래스를 실행해서 전체 프로세스를 한 번에 수행해봅시다!

In [ ]:
"""
s3_LegalChunkingStrategy.py
법령 문서 청킹

메타데이터 구조:
- doc_id, doc_name, page: 문서 식별
- chunk_tokens: 토큰 수
- has_overlap: 오버랩 여부
"""

import re
import json
import os
from typing import List, Dict
import tiktoken


class LegalChunkingStrategy:
    """법령 청킹 전략"""
    
    def __init__(self, chunk_size: int = 800, overlap: int = 200, model: str = "gpt-4"):
        """
        Args:
            chunk_size: 청크 최대 토큰 수
            overlap: 청크 간 오버랩 토큰 수
            model: 토큰 계산용 모델명
        """
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.encoder = tiktoken.encoding_for_model(model)

        print(f"📐 간소화 청킹 전략 초기화")        
        print(f"  - 청크 크기: {chunk_size} 토큰")
        print(f"  - 오버랩: {overlap} 토큰")
        print(f"  - 메타데이터: 최소화 (doc 정보만)")
    
    def count_tokens(self, text: str) -> int:
        """텍스트의 토큰 수 계산"""
        return len(self.encoder.encode(text))
    
    def split_by_tokens(self, text: str, max_tokens: int) -> List[Dict]:
        """
        토큰 단위로 강제 분할 (최후의 수단)
        800토큰씩 잘라서 반환
        """
        tokens = self.encoder.encode(text)
        chunks = []
        
        for i in range(0, len(tokens), max_tokens):
            chunk_tokens = tokens[i:i + max_tokens]
            chunk_text = self.encoder.decode(chunk_tokens)
            chunks.append({"text": chunk_text})
        
        return chunks
    
    def split_by_article(self, text: str, max_tokens: int) -> List[Dict]:
        """
        조(條) 단위로 텍스트 분할
        
        분할 우선순위:
        1. 조(條) 단위
        2. 항(項) 단위  
        3. 토큰 제한
        """
        # 제○조 패턴으로 분할
        parts = re.split(r'(제\d+조(?:의\d+)?)', text)
        
        chunks = []
        current_chunk = ""
        current_tokens = 0
        
        i = 0
        while i < len(parts):
            part = parts[i].strip()
            
            if not part:
                i += 1
                continue
            
            # 제○조 패턴
            if re.match(r'제\d+조(?:의\d+)?$', part):
                article = part
                content = ""
                
                if i + 1 < len(parts):
                    content = parts[i + 1].strip()
                
                full_text = article + " " + content
                text_tokens = self.count_tokens(full_text)
                
                # 조 전체가 max_tokens 이하
                if text_tokens <= max_tokens:
                    if current_tokens + text_tokens <= max_tokens:
                        # 현재 청크에 추가
                        current_chunk += (" " if current_chunk else "") + full_text
                        current_tokens += text_tokens
                    else:
                        # 현재 청크 저장
                        if current_chunk:
                            chunks.append({"text": current_chunk})
                        # 새 청크 시작
                        current_chunk = full_text
                        current_tokens = text_tokens
                else:
                    # 조가 너무 크면 항 단위로 분할
                    if current_chunk:
                        chunks.append({"text": current_chunk})
                    
                    para_chunks = self._split_by_paragraph(article, content, max_tokens)
                    chunks.extend(para_chunks)
                    
                    current_chunk = ""
                    current_tokens = 0
                
                i += 2
            else:
                # 일반 텍스트
                part_tokens = self.count_tokens(part)
                
                if part_tokens > max_tokens:
                    # 현재 청크 저장
                    if current_chunk:
                        chunks.append({"text": current_chunk})
                    
                    # 토큰 단위로 강제 분할
                    token_chunks = self.split_by_tokens(part, max_tokens)
                    chunks.extend(token_chunks)
                    
                    # 새 청크 시작
                    current_chunk = ""
                    current_tokens = 0
                elif current_tokens + part_tokens <= max_tokens:
                    current_chunk += (" " if current_chunk else "") + part
                    current_tokens += part_tokens
                else:
                    if current_chunk:
                        chunks.append({"text": current_chunk})
                    current_chunk = part
                    current_tokens = part_tokens
                
                i += 1
        
        # 마지막 청크
        if current_chunk:
            chunks.append({"text": current_chunk})
        
        return chunks
    
    def _split_by_paragraph(self, article: str, content: str, max_tokens: int) -> List[Dict]:
        """항 단위로 분할"""
        # ①②③ 또는 1. 2. 3. 패턴으로 분할
        parts = re.split(r'([①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]|\d+\.)', content)
        
        chunks = []
        current_chunk = article + " "
        current_tokens = self.count_tokens(current_chunk)
        
        for part in parts:
            if not part.strip():
                continue
            
            part_tokens = self.count_tokens(part)
            
            if part_tokens > max_tokens:
                # 현재 청크 저장
                if current_chunk.strip() != article:
                    chunks.append({"text": current_chunk.strip()})
                
                # 항을 토큰 단위로 강제 분할
                token_chunks = self.split_by_tokens(article + " " + part, max_tokens)
                chunks.extend(token_chunks)
                
                # 새 청크 시작
                current_chunk = article + " "
                current_tokens = self.count_tokens(current_chunk)
            elif current_tokens + part_tokens <= max_tokens:
                current_chunk += part + " "
                current_tokens += part_tokens
            else:
                if current_chunk.strip() != article:
                    chunks.append({"text": current_chunk.strip()})
                current_chunk = article + " " + part + " "
                current_tokens = self.count_tokens(current_chunk)
        
        if current_chunk.strip() != article:
            chunks.append({"text": current_chunk.strip()})
        
        return chunks
    
    def process_from_unified_json(self, json_path: str) -> List[Dict]:
        """
        통합 JSON에서 청킹 수행
        
        Returns:
            청크 리스트 with 간소화된 메타데이터
        """
        print(f"\n📖 JSON 로드: {json_path}")
        
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        text_blocks = data.get('text_blocks', [])
        print(f"✓ {len(text_blocks)}개 텍스트 블록")
        
        all_chunks = []
        chunk_counter = 1
        
        print(f"\n🔪 청킹 시작...")
        
        for idx, block in enumerate(text_blocks, 1):
            text = block['text']
            
            # 구조 인식 청킹
            text_chunks = self.split_by_article(text, self.chunk_size)
            
            for chunk_data in text_chunks:
                chunk_text = chunk_data["text"]
                
                # 간소화된 메타데이터
                chunk = {
                    "chunk_id": f"chunk_{chunk_counter:05d}",
                    "content": chunk_text,
                    "metadata": {
                        "doc_id": block["doc_id"],
                        "doc_name": block["doc_name"],
                        "page": block.get("page", 0),
                        "chunk_tokens": self.count_tokens(chunk_text)
                    }
                }
                all_chunks.append(chunk)
                chunk_counter += 1
            
            if idx % 50 == 0:
                print(f"  진행: {idx}/{len(text_blocks)} 블록...")
        
        # 오버랩 적용
        print(f"\n🔗 오버랩 적용...")
        final_chunks = self.apply_overlap(all_chunks)
        
        print(f"✅ 최종 {len(final_chunks)}개 청크\n")
        
        return final_chunks
    
    def apply_overlap(self, chunks: List[Dict]) -> List[Dict]:
        """청크 간 오버랩 적용"""
        if not chunks or self.overlap == 0:
            return chunks
        
        overlapped_chunks = []
        
        for i, chunk in enumerate(chunks):
            content = chunk["content"]
            
            # 다음 청크가 있고, 같은 문서인 경우
            if i < len(chunks) - 1:
                next_chunk = chunks[i + 1]
                
                if next_chunk["metadata"]["doc_id"] == chunk["metadata"]["doc_id"]:
                    next_content = next_chunk["content"]
                    
                    # 오버랩 토큰 추출
                    tokens = self.encoder.encode(next_content)
                    if len(tokens) > self.overlap:
                        overlap_tokens = tokens[:self.overlap]
                        overlap_text = self.encoder.decode(overlap_tokens)
                        content += f"\n\n[다음 내용 미리보기]\n{overlap_text}..."
            
            # 오버랩 적용된 청크
            overlapped_chunks.append({
                **chunk,
                "content": content,
                "metadata": {
                    **chunk["metadata"],
                    "has_overlap": i < len(chunks) - 1,
                    "chunk_tokens": self.count_tokens(content)
                }
            })
        
        return overlapped_chunks
    
    def save_chunks(self, chunks: List[Dict], output_path: str):
        """청크를 JSON 파일로 저장"""
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(chunks, f, ensure_ascii=False, indent=2)
        
        print(f"💾 청크 저장: {output_path}")
        print(f"  - 청크 수: {len(chunks)}개")


def main():
    """실행 메인 함수"""
    print("="*80)
    print("🔪 간소화 법령 청킹")
    print("="*80)
    
    # 경로 설정
    current_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.dirname(current_dir)
    
    INPUT_PATH = os.path.join(project_root, "data", "processed", "construction_law_unified.json")
    OUTPUT_PATH = os.path.join(project_root, "data", "chunks", "construction_law_chunks.json")
    
    print(f"\n입력: {INPUT_PATH}")
    print(f"출력: {OUTPUT_PATH}")
    
    if not os.path.exists(INPUT_PATH):
        print(f"\n✗ 입력 파일 없음: {INPUT_PATH}")
        return
    
    if os.path.exists(OUTPUT_PATH):
        response = input(f"\n⚠ 파일 존재. 덮어쓰기? (y/n): ")
        if response.lower() != 'y':
            print("취소")
            return
    
    try:
        chunker = LegalChunkingStrategy(
            chunk_size=800,
            overlap=200,
            model="gpt-4"
        )
        
        chunks = chunker.process_from_unified_json(INPUT_PATH)
        chunker.save_chunks(chunks, OUTPUT_PATH)
        
        print("="*80)
        print("✅ 청킹 완료!")
        print("="*80)
        
    except Exception as e:
        print(f"\n✗ 오류: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()